# CLM-0.4-mini M1 Calibration

**Development calibration only.** This notebook opens seed `90401` and must never run formal seeds `90411/90412/90413`. It selects the first passing optimizer configuration in the pre-registered order; it does not emit an M1 scientific decision.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/ArcheLabs/mini-cells.git'
REPO_REF = 'main'
ROOT = Path('/kaggle/working/mini-cells')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',REPO_REF,'--depth','1'], check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout',REPO_REF], check=True)
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',REPO_REF], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-e','.[dev,lm]','-q'], cwd=ROOT, check=True)
subprocess.run(['git','-C',str(ROOT),'status','--short'], check=True)


In [ ]:
DATA = Path('/kaggle/working/clm-0.4-mini-data')
OUT = Path('/kaggle/working/clm-0.4-mini-calibration')
assert (DATA / 'asset-summary.json').is_file(), 'Run formal data preparation first.'
import json
assets = json.loads((DATA / 'asset-summary.json').read_text())
assets


In [ ]:
# Validate the pre-registered plan without observing development seed 90401.
PLAN_OUT = Path('/kaggle/working/clm-0.4-mini-calibration-plan-check')
subprocess.run([sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-mini-calibration', '--plan-only', '--out', str(PLAN_OUT)], cwd=ROOT, check=True)
plan = json.loads((PLAN_OUT/'calibration-plan.json').read_text())
assert plan['candidate_count'] == 81
plan['plan_sha256'], plan['candidates'][0], plan['candidates'][-1]


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before calibration.'
torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda


## Open development seed 90401

The next cell is the **first allowed observation of development seed `90401`**. It trains the ~5M base model once on the frozen 30M-token corpus, checks base prerequisites, then evaluates candidates in the committed order and stops at the first full development-gate pass. The run is resumable from `OUT` after interruption.

Do not edit the grid, gates, asset hashes, curriculum, routing salt, or candidate order after running this cell.

In [ ]:
subprocess.run([
    sys.executable, str(ROOT/'scripts/research/run.py'), 'clm-0.4-mini-calibration',
    '--data-dir', str(DATA), '--out', str(OUT), '--device', 'cuda',
    '--seed', '90401', '--confirm-development-seed', '90401',
], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT/'scripts/research/report.py'), 'clm-0.4-mini-calibration', '--results', str(OUT)], cwd=ROOT, check=True)


In [ ]:
decision = json.loads((OUT/'decision.json').read_text())
assert decision['scientific_decision'] is False
assert decision['development_seed_observed'] is True
assert decision['formal_seeds_observed'] is False
decision


In [ ]:
if decision['status'] == 'CALIBRATION_CONFIGURATION_SELECTED':
    selected = json.loads((OUT/'selected.json').read_text())
    lock = json.loads((OUT/'protocol-lock.candidate.json').read_text())
    print(json.dumps(selected['candidate'], indent=2))
    print('Protocol lock candidate:', OUT/'protocol-lock.candidate.json')
else:
    print('No formal seed may be opened. Status:', decision['status'])


## Next boundary

If and only if calibration selects a configuration, review `protocol-lock.candidate.json` and commit it as the canonical `research/validations/clm-0.4-mini-language-validation/protocol-lock.json` in a separate lock step. **Formal seeds remain forbidden until that commit exists.**